# Titanic ⑥ パラメータを調整する

05 で選んだ勾配ブースティングのパラメータを，Optuna で調整した記録．05 の既定値では，交差検証の正解率 0.820 に対して train での正解率が 0.947 で，過学習していた．

| 項目 | 決めたこと |
|---|---|
| 調整するパラメータ | 学習率 `learning_rate`，木の本数 `max_iter`，正則化 `l2_regularization`（木の大きさと葉の最小人数は既定値のまま） |
| 探す範囲 | 学習率 0.01〜0.3（対数の目盛り），木の本数 50〜500，正則化 0〜10 |
| 探し方 | Optuna で 50 回（乱数の種は 0） |
| 評価 | 5 分割交差検証を 5 回繰り返した正解率の平均（04 以降と同じ） |
| 最終的なスコア | 探索で最高だった平均をそのまま使う（50 回の中から選んだ分だけ楽観的になることは承知のうえ） |

Optuna の既定の探し方（TPE）は，それまでの試行を「スコアが良かったもの」と「それ以外」に分け，良かったものが集まっている辺りを重点的に試していく．最初の 10 回はランダムに試す．

In [ ]:
# src/ 以下を編集したら，カーネルを再起動しなくても読み込み直す
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import optuna
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_validate

from titanic.data import load_data, save_submission
from titanic.features import CATEGORICAL, IMPUTE_BY, NUMERIC, fill_embarked
from titanic.modeling import CV, build_pipeline, cv_accuracy
from titanic.visualize import BASELINE, INK_MUTED, SURFACE, set_style

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

train, test = load_data()
X, y = fill_embarked(train), train["Survived"]
X_test = fill_embarked(test)

## 探索

50 回で 1 分弱かかる．

In [ ]:
SEARCH_SPACE = {
    "learning_rate": (0.01, 0.3),
    "max_iter": (50, 500),
    "l2_regularization": (0.0, 10.0),
}


def objective(trial: optuna.Trial) -> float:
    """試行ごとにパラメータを選び，交差検証の正解率の平均を返す．"""
    params = {
        "learning_rate": trial.suggest_float("learning_rate", *SEARCH_SPACE["learning_rate"], log=True),
        "max_iter": trial.suggest_int("max_iter", *SEARCH_SPACE["max_iter"]),
        "l2_regularization": trial.suggest_float("l2_regularization", *SEARCH_SPACE["l2_regularization"]),
    }
    pipeline = build_pipeline(HistGradientBoostingClassifier(**params), NUMERIC, CATEGORICAL, IMPUTE_BY)
    return cv_accuracy(pipeline, X, y).mean()


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=50)
print(f"最高: 試行 {study.best_trial.number}，交差検証の平均 {study.best_value:.4f}")
print(study.best_params)

In [ ]:
trials = study.trials_dataframe()[["number", "value", "params_learning_rate", "params_max_iter", "params_l2_regularization"]]
trials.columns = ["試行", "CV 平均", "learning_rate", "max_iter", "l2_regularization"]
print("上位 10 件")
print(trials.nlargest(10, "CV 平均").round(4).to_string(index=False))
print()
print("下位 5 件")
print(trials.nsmallest(5, "CV 平均").round(4).to_string(index=False))

上位 10 件は，学習率 0.019〜0.038，木の本数 343〜488，正則化 6.4〜10.0 に集まった．「学習率を下げて木を多めに足し，強めに正則化する」組み合わせで，上位どうしの差は 0.0015 しかない．

下位には 2 種類の失敗が見える．学習率 0.013 で木が 59 本の試行（0.789）は，学習が足りていない（未学習）．学習率 0.069 で木が 467 本，正則化が 0.7 の試行（0.807）は，学習が進みすぎて過学習している．

## 各パラメータと交差検証の正解率の関係

1 点が 1 回の試行．横軸の範囲は探索した範囲そのままにしてある．

In [ ]:
set_style()
BEST_COLOR = "#2a78d6"
best = trials.loc[trials["CV 平均"].idxmax()]
default_cv = 0.820  # 05 の既定値での交差検証の平均

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, (name, (low, high)) in zip(axes, SEARCH_SPACE.items()):
    # 範囲の端の点が枠で切れないように，点だけは枠の外まで描く
    ax.scatter(trials[name], trials["CV 平均"], s=40, color=INK_MUTED, edgecolor=SURFACE, linewidth=1.5, clip_on=False)
    ax.scatter(best[name], best["CV 平均"], s=90, color=BEST_COLOR, edgecolor=SURFACE, linewidth=1.5, zorder=3, clip_on=False)
    ax.axhline(default_cv, color=BASELINE, linewidth=1)
    if name == "learning_rate":
        ax.set_xscale("log")
        ax.minorticks_off()
        ax.set_xticks([0.01, 0.03, 0.1, 0.3], ["0.01", "0.03", "0.1", "0.3"])
    ax.set_xlim(low, high)
    ax.set_title(name)
axes[0].set_ylabel("交差検証の正解率")
fig.suptitle("各試行の交差検証の正解率（青: 最高の試行，横線: 既定値の 0.820）", x=0.01, ha="left", fontsize=12)
plt.show()

学習率は 0.02〜0.04 あたりが山になっていて，0.1 を超えると下がる．木の本数は多いほど，正則化は強いほど良い傾向があり，良い試行は範囲の右端（木の本数 500，正則化 10）に寄っている．範囲を広げれば，もっと良い値が見つかるかもしれない．

## 既定値との比較

train での正解率も並べて，過学習がどう変わったかを見る．

In [ ]:
settings = {"既定値（05）": {}, f"最高の試行（{study.best_trial.number}）": study.best_params}
rows = {}
for name, params in settings.items():
    pipeline = build_pipeline(HistGradientBoostingClassifier(**params), NUMERIC, CATEGORICAL, IMPUTE_BY)
    result = cross_validate(pipeline, X, y, cv=CV, return_train_score=True, n_jobs=-1)
    rows[name] = {
        "CV 平均": result["test_score"].mean(),
        "CV 標準偏差": result["test_score"].std(),
        "train 平均": result["train_score"].mean(),
        "train との差": result["train_score"].mean() - result["test_score"].mean(),
    }
print(pd.DataFrame(rows).T.round(3))

交差検証の正解率は 0.820 から 0.833 に上がり，train との差は 0.127 から 0.068 に縮んだ．ただし 0.833 は 50 回の試行の中で最高だった値なので，選んだ分だけ楽観的になっている．

**決定: 試行 25 の設定（learning_rate 0.0256，max_iter 456，l2_regularization 7.99）を使う．**

## 提出ファイルを作る

交差検証の各回のモデルは train の 4/5 しか見ていないので，最後は train 全体（891 行）で学習し直して test を予測する．

In [ ]:
final_model = build_pipeline(HistGradientBoostingClassifier(**study.best_params), NUMERIC, CATEGORICAL, IMPUTE_BY).fit(X, y)
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": final_model.predict(X_test),
})
path = save_submission(submission, "06_tuning")
print("書き出し先:", path)
print(f"test での予測生存率: {submission['Survived'].mean():.3f}（train の生存率は 0.384）")